# 05 - Theme first derivative (catch the take-off)

Identical logic to notebook 03, but operating on **themes** instead of individual
tickers. Reads `data/processed/daily_theme_counts.parquet` (produced by notebook 04).

**The idea:** smooth the daily keyword-hit signal, then take the first derivative
(how much did it change since yesterday?). When that speed jumps well above normal,
a theme is catching fire in retail discussion. We mark those days in red.

Set `VALUE_COLUMN = 'weighted_count'` to use the upvote-weighted signal instead.

In [ ]:
import os, sys
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
print('project root:', ROOT)

In [ ]:
# ============ PARAMETERS - edit these ============
DAILY_THEMES_PATH = os.path.join(ROOT, 'data', 'processed', 'daily_theme_counts.parquet')
VALUE_COLUMN      = 'mention_count'   # 'mention_count' (raw hits) or 'weighted_count' (by upvotes)
THEMES            = []     # e.g. ['memory', 'ai']; [] = use the TOP_N most mentioned
TOP_N             = 4
SMOOTH            = 3      # rolling-average window in days
K                 = 2.0    # std-devs above normal that counts as a take-off
# ==================================================

In [ ]:
# ============ TIME WINDOW - edit freely ============
# Clips the daily counts before the analysis. None = keep everything in the
# counts file (which was already windowed when notebook 02/04 produced it).
START_DATE = None   # inclusive,  'YYYY-MM-DD' or None
END_DATE   = None   # EXCLUSIVE,  'YYYY-MM-DD' or None
# ====================================================

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.inflection import build_daily_series, compute_inflection

daily = pd.read_parquet(DAILY_THEMES_PATH)
daily['date'] = pd.to_datetime(daily['date'])

# Apply the time window from the cell above.
if START_DATE:
    daily = daily[daily['date'] >= pd.Timestamp(START_DATE)]
if END_DATE:
    daily = daily[daily['date'] < pd.Timestamp(END_DATE)]
print('window:', START_DATE, 'to', END_DATE, '|', len(daily), 'daily rows kept')

# inflection.py expects a 'ticker' column; rename 'theme' so it works unchanged.
daily = daily.rename(columns={'theme': 'ticker'})

if VALUE_COLUMN not in daily.columns:
    print('NOTE:', VALUE_COLUMN, 'not found - using mention_count instead.')
    VALUE_COLUMN = 'mention_count'

if THEMES:
    chosen = THEMES
else:
    totals = daily.groupby('ticker')[VALUE_COLUMN].sum().sort_values(ascending=False)
    chosen = list(totals.head(TOP_N).index)
print('analysing:', chosen, '| signal:', VALUE_COLUMN)

In [ ]:
for theme in chosen:
    series = build_daily_series(daily, theme, value_col=VALUE_COLUMN)
    if series is None:
        print('skip', theme, '(not found)'); continue
    result = compute_inflection(series, SMOOTH, K)
    flags = result[result['is_inflection']]

    print(f'\n{theme} ({VALUE_COLUMN}): {len(flags)} inflection day(s)')
    for date, row in flags.iterrows():
        print('   ', date.strftime('%Y-%m-%d'), '| value', int(row['count']), '| velocity', round(row['velocity'], 1))

    fig, (top, bottom) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
    top.plot(result.index, result['count'], color='lightgray', label='raw')
    top.plot(result.index, result['smoothed'], color='steelblue', linewidth=2, label='smoothed')
    top.scatter(flags.index, flags['smoothed'], color='red', zorder=5, label='inflection day')
    top.set_title(f'{theme} - {VALUE_COLUMN}')
    top.legend(); top.grid(True, alpha=0.3)

    bottom.plot(result.index, result['velocity'], color='darkorange', label='first derivative')
    bottom.axhline(result.attrs['threshold'], color='red', linestyle='--', label='spike threshold')
    bottom.axhline(0, color='black', linewidth=0.6)
    bottom.scatter(flags.index, flags['velocity'], color='red', zorder=5)
    bottom.set_title('first derivative - how fast the theme is growing')
    bottom.legend(); bottom.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()